In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
# Job parameters — inherited from bundle resources via job parameter defaults.
# These ensure the notebook always operates on the actual deployed infrastructure.
dbutils.widgets.text("lakebase_project_id", "hi-genie-orchestrator", "Lakebase Project ID")
dbutils.widgets.text("branches", "production,development", "Branches (comma-separated)")
dbutils.widgets.text("database_name", "databricks-postgres", "Database Name")
dbutils.widgets.text("catalog", "", "UC Catalog (for logging)")
dbutils.widgets.text("schema", "", "UC Schema (for logging)")

# Read parameters
lakebase_project_id = dbutils.widgets.get("lakebase_project_id")
branches = [b.strip() for b in dbutils.widgets.get("branches").split(",") if b.strip()]
database_name = dbutils.widgets.get("database_name")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Lakebase Project: {lakebase_project_id}")
print(f"Branches:         {branches}")
print(f"Database:         {database_name}")
print(f"UC Location:      {catalog}.{schema}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
results = {}

for branch in branches:
    db_path = f"projects/{lakebase_project_id}/branches/{branch}/databases/{database_name}"
    print(f"\n{'='*60}")
    print(f"Branch: {branch}")
    print(f"Database path: {db_path}")
    print(f"{'='*60}")

    # ── Step 1: Check if Lakebase Search is enabled (extensions available) ───
    check_sql = "SELECT name FROM pg_available_extensions WHERE name IN ('lakebase_vector', 'lakebase_text') ORDER BY name;"
    try:
        check_result = w.lakebase.execute_statement(database=db_path, statement=check_sql)
        available_extensions = [row[0] for row in (check_result.result.data_array or [])]
    except Exception as e:
        # Fallback: try alternate SDK method name
        try:
            check_result = w.postgres.execute_sql(database=db_path, sql_statement=check_sql)
            available_extensions = [row[0] for row in (check_result.result.data_array or [])]
        except Exception as e2:
            print(f"  ERROR: Could not query pg_available_extensions: {e2}")
            results[branch] = {"status": "error", "message": str(e2)}
            continue

    if not available_extensions:
        error_msg = (
            f"  ✗ FAILED: Lakebase Search is NOT enabled for project '{lakebase_project_id}'.\n"
            f"\n"
            f"  This is a manual, one-time operation (no API/DABs method exists):\n"
            f"    1. Open the Lakebase App UI\n"
            f"    2. Navigate to project '{lakebase_project_id}' → Settings → Search\n"
            f"    3. Click 'Enable Search'\n"
            f"\n"
            f"  WARNING: This is irreversible and may restart computes/drop connections.\n"
            f"  Re-run this job after enabling."
        )
        print(error_msg)
        results[branch] = {"status": "failed", "message": "Lakebase Search not enabled"}
        raise RuntimeError(f"Lakebase Search not enabled for project '{lakebase_project_id}'. See output above for remediation steps.")

    print(f"  ✓ Lakebase Search is enabled. Available extensions: {available_extensions}")

    # ── Step 2: Install extensions (idempotent) ──────────────────────────────
    extensions_to_install = [
        ("lakebase_vector", "CREATE EXTENSION IF NOT EXISTS lakebase_vector CASCADE;"),
        ("lakebase_text", "CREATE EXTENSION IF NOT EXISTS lakebase_text;"),
    ]

    installed = []
    for ext_name, ext_sql in extensions_to_install:
        try:
            try:
                w.lakebase.execute_statement(database=db_path, statement=ext_sql)
            except AttributeError:
                w.postgres.execute_sql(database=db_path, sql_statement=ext_sql)
            print(f"  ✓ {ext_name} — installed (or already exists)")
            installed.append(ext_name)
        except Exception as e:
            print(f"  ✗ {ext_name} — FAILED: {e}")
            results[branch] = {"status": "partial", "installed": installed, "error": str(e)}
            raise

    results[branch] = {"status": "success", "installed": installed}

In [0]:
# Verify extensions are actually installed (appear in pg_extension, not just available)
verification = {}

for branch in branches:
    db_path = f"projects/{lakebase_project_id}/branches/{branch}/databases/{database_name}"
    verify_sql = "SELECT extname, extversion FROM pg_extension WHERE extname IN ('lakebase_vector', 'lakebase_text', 'vector') ORDER BY extname;"

    try:
        try:
            result = w.lakebase.execute_statement(database=db_path, statement=verify_sql)
        except AttributeError:
            result = w.postgres.execute_sql(database=db_path, sql_statement=verify_sql)

        installed_exts = {row[0]: row[1] for row in (result.result.data_array or [])}
        verification[branch] = installed_exts

        print(f"\n{branch}:")
        if installed_exts:
            for ext, ver in sorted(installed_exts.items()):
                print(f"  ✓ {ext} v{ver}")
        else:
            print(f"  ✗ No extensions found in pg_extension!")

    except Exception as e:
        print(f"\n{branch}: ERROR verifying — {e}")
        verification[branch] = {"error": str(e)}

In [0]:
# Final summary
print("\n" + "="*60)
print("LAKEBASE PLATFORM SETUP — SUMMARY")
print("="*60)
print(f"Project:  {lakebase_project_id}")
print(f"Database: {database_name}")
print(f"UC Ref:   {catalog}.{schema}")
print()

all_success = True
for branch in branches:
    status = results.get(branch, {}).get("status", "unknown")
    symbol = "✓" if status == "success" else "✗"
    print(f"  {symbol} {branch}: {status}")
    if status != "success":
        all_success = False

print()
if all_success:
    print("All branches configured successfully.")
    print("Extensions installed: lakebase_vector (pgvector ANN), lakebase_text (BM25)")
else:
    print("WARNING: Some branches had issues. Review output above.")
    # Don't fail the job here — earlier cells raise on critical errors